# Pre-Interview Assignment - Gabriela Janiszewska

## Loading my data


The dataset represents crime incidents reported across different districts and cities. Each row corresponds to a single incident with details about the crime, suspect information, reporting method, and investigation status. For this imaginary scenario, let's assume we are creating a crime analysis for the Ministry of Justice.
I chose the Crime Incidents dataset from Kaggle because it demonstrates real-world data quality challenges (multiple date formats, duplicates, invalid values) that are common in operational systems and showcase comprehensive data cleaning skills.

In [0]:
df = spark.read.csv("/Workspace/Users/gabrielajaniszewska@translite.pl/crime_incidents_messy.csv", header = True, inferSchema = True)
df.show(10, truncate = False)

In [0]:
# Number of rows and columns in the dataset
print(f"Number of rows: {df.count()}, number of columns: {len(df.columns)}")


## Initial Data Analysis


In [0]:
# Checking the schema
df.printSchema()

Since the dataset contains personal identifiable information (PII) such as first names, last names and id, I am dropping these columns at the very start for confidentiality reasons.

In [0]:
# Dropping PII columns
cols_to_drop = ["officer_first_name", "officer_last_name", "officer_id", "badge_number",
                "suspect_first_name", "suspect_last_name", "suspect_id",
                "victim_first_name", "victim_last_name", "victim_id", "victim_phone",
                "address", "notes"]
# suspect_race and victim_gender are retained as sensitive but necessary attributes for bias detection and crime pattern analysis.
df = df.drop(*cols_to_drop)

In [0]:
# The incident_datetime is a string. Changing it to datetime format
# Checking unique values in incident_datetime column to see the format:
from pyspark.sql.functions import col
df.select(col("incident_datetime")).distinct().show(150, truncate=False)

In [0]:
from pyspark.sql.functions import regexp_extract
# Checking the data format using regex to confirm the type of abnormalities
df.select(
    regexp_extract("incident_datetime", r"^\d{1,4}([-/])", 1).alias("separator"),
    regexp_extract("incident_datetime", r"^\d{1,4}[-/]\d{1,2}[-/]\d{2,4}", 0).alias("date_pattern")
).distinct().show(100, truncate=False)

In [0]:
# The incident_datetime column has several formats, so I will need to use coalesce function
from pyspark.sql.functions import try_to_timestamp, coalesce, lit, col

df = df.withColumn("incident_datetime", 
    coalesce(
        try_to_timestamp(col("incident_datetime"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("incident_datetime"), lit("MM-dd-yyyy")),
        try_to_timestamp(col("incident_datetime"), lit("dd-MM-yyyy")),
        try_to_timestamp(col("incident_datetime"), lit("MM/dd/yyyy HH:mm"))
    )
)
# Checking the boundaries of the datetime column
df.select("incident_datetime").agg({"incident_datetime": "min"}).show()
df.select("incident_datetime").agg({"incident_datetime": "max"}).show()

# Checking the schema
df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Dividing columns into type groups to count the nulls and NA
string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
numeric_cols = [f.name for f in df.schema.fields if str(f.dataType) in ["DoubleType()", "IntegerType()"]]
other_cols = [f.name for f in df.schema.fields if f.name not in string_cols + numeric_cols]

print("String columns:", string_cols)
print("Numeric columns:", numeric_cols)
print("Other (timestamp, bool, etc.):", other_cols)


In [0]:
from pyspark.sql.functions import isnan

# Strings - isNull only
df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in string_cols]).show(truncate=False)


In [0]:

# Numeric - isNull and isnan
df.select([spark_sum(when(col(c).isNull() | isnan(c), 1).otherwise(0)).alias(c) for c in numeric_cols]).show(truncate=False)

# Malformed values are causing errors that need to be investigated

In [0]:
# Remaining columns (timestamp, boolean) - only isNull
df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in other_cols]).show(truncate=False)

In [0]:
# Checking the unique values in reported_online column
df.select(col("reported_online")).distinct().show()

In [0]:
# Changing the type of reported_online column to boolean taking into account unique values
from pyspark.sql.functions import when, lower, trim

df = df.withColumn("reported_online",
    when(lower(trim(col("reported_online"))).isin("true", "yes", "1"), True)
    .when(lower(trim(col("reported_online"))).isin("false", "no", "0"), False)
    .otherwise(None)
    .cast("boolean")
)

In [0]:
# Checking the num_arrests unique values
df.select(col("num_arrests")).distinct().show()

In [0]:
# Fixing the num_arrests values and casting to integer
from pyspark.sql.functions import when, col

df = df.withColumn("num_arrests_flag", 
    when(col("num_arrests") < 0, "NEEDS_REVIEW") # Flagging negative values for review in the source
    .otherwise(None)
) \
.withColumn("num_arrests",
    when(col("num_arrests").isNull(), 0)
    .when(col("num_arrests") < 0, None)
    .otherwise(col("num_arrests"))
    .cast("integer")
)

In [0]:
# Checking the property_loss_usd column
df.select(col("property_loss_usd")).distinct().show(100)


In [0]:
from pyspark.sql.functions import when, col, regexp_replace

# Step 1: clean malformed values (e.g. "46792.6.0") and cast to double
df = df.withColumn("property_loss_usd",
    regexp_replace(col("property_loss_usd"), r"(\.\d+)\.0$", "$1")
).withColumn("property_loss_usd",
    col("property_loss_usd").cast("double")
)

# Step 2: flag negatives and replace with null
df = df.withColumn("property_loss_usd_flag",
    when(col("property_loss_usd") < 0, "NEEDS_REVIEW")
    .otherwise(None)
).withColumn("property_loss_usd",
    when(col("property_loss_usd") < 0, None)
    .otherwise(col("property_loss_usd"))
)


In [0]:
# Checking the case_status column
df.select("case_status").distinct().show()


In [0]:
# Fixing the case_status column
from pyspark.sql.functions import when, lower, trim

df = df.withColumn("case_status",
    when(lower(trim(col("case_status"))).isin("open"), "Open")
    .when(lower(trim(col("case_status"))).isin("closed"), "Closed")
    .when(lower(trim(col("case_status"))).isin("pending", "pendng"), "Pending")
    .when(lower(trim(col("case_status"))).isin("under investigation", "investgation"), "Under Investigation")
    .when(lower(trim(col("case_status"))).isin("resolved"), "Resolved")
    .otherwise("Unknown")
)


In [0]:
# Checking the resolution column
df.select("resolution").distinct().show()

In [0]:
# Fixing the resolution column
df = df.withColumn("resolution",
    when(lower(trim(col("resolution"))).isin("arrest made", "arres made"), "Arrest Made")
    .when(lower(trim(col("resolution"))).isin("no arrest"), "No Arrest")
    .when(lower(trim(col("resolution"))).isin("dismissed", "case dismissed"), "Case Dismissed")
    .when(lower(trim(col("resolution"))).isin("warning", "warning issued"), "Warning Issued")
    .otherwise("Unknown")  # N/A and NULL → "Unknown"
)

In [0]:
# Checking the suspect_age and victim_age column
df.select("suspect_age").distinct().show()
df.select("victim_age").distinct().show()


In [0]:
# The suspect_age and victim_age columns has some negative values and many unrealistic values. I will retain only the realistic data and turn it into a categorical column
df = df.withColumn("suspect_age",
    when((col("suspect_age") >= 0) & (col("suspect_age") <= 120), col("suspect_age"))
    .otherwise(None)
)

df = df.withColumn("suspect_age_group",
    when(col("suspect_age").isNull(), "Unknown")
    .when(col("suspect_age") < 18, "Minor")
    .when(col("suspect_age") < 30, "Young Adult")
    .when(col("suspect_age") < 60, "Adult")
    .otherwise("Senior")
)

# Dropping suspect_age column
df = df.drop("suspect_age")

df = df.withColumn("victim_age",
    when((col("victim_age") >= 0) & (col("victim_age") <= 120), col("victim_age"))
    .otherwise(None)
)

df = df.withColumn("victim_age_group",
    when(col("victim_age").isNull(), "Unknown")
    .when(col("victim_age") < 18, "Minor")
    .when(col("victim_age") < 30, "Young Adult")
    .when(col("victim_age") < 60, "Adult")
    .otherwise("Senior")
)

df = df.drop("victim_age")


In [0]:
# Checking the suspect_gender and victim_gender columns
df.select("suspect_gender").distinct().show()
df.select("victim_gender").distinct().show()

In [0]:
# Fixing the suspect_gender and victim_gender columns
from pyspark.sql.functions import when, trim, lower

def standardize_gender(column):
    return (
        when(lower(trim(col(column))).isin("male", "m"), "Male")
        .when(lower(trim(col(column))).isin("female", "f"), "Female")
        .when(lower(trim(col(column))).isin("other"), "Other")
        .otherwise(None)  # N/A, Unknown, NULL → null
    )

df = df.withColumn("suspect_gender", standardize_gender("suspect_gender")) \
       .withColumn("victim_gender", standardize_gender("victim_gender"))

In [0]:
df.printSchema()

In [0]:
# Re-checking the number of null values in each column
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in df.columns
])
null_counts.show()

### Addressing the remaining null values
Leaving null values: num_arrests, num_arrests_flag, property_loss_usd, property_loss_usd_flag - as intended
Leaving null values: reported_online, suspect_gender, suspect_race, victim_gender, weapon_used - as intended, no data available



In [0]:
# Checking the data for duplicates
from pyspark.sql.functions import count

total = df.count()
distinct = df.dropDuplicates(["incident_id"]).count()
print(f"Duplicate rows: {total - distinct}")

df.groupBy("incident_id") \
  .count() \
  .filter(col("count") > 1) \
  .show()

In [0]:
# Dropping duplicates - keeping one row per incident_id
df_deduplicated = df.dropDuplicates(["incident_id"])

# Checking the number of rows
print(f"Number of rows after deduplication: {df_deduplicated.count()}")



In [0]:
# Filtering to closed cases with confirmed arrests for analysis
df_filtered = df_deduplicated.filter(
    (col("case_status") == "Closed") & (col("num_arrests") > 0)
)
df_filtered.show(10, truncate=False)

## SQL Analysis

In [0]:
# Registering as temp view
df_filtered.createOrReplaceTempView("crime_incidents")

# Querying: suspect_gender breakdown
result = spark.sql("""
    SELECT 
        suspect_gender,
        resolution,
        COUNT(*) AS incident_count,
        ROUND(AVG(property_loss_usd), 2) AS avg_property_loss_usd
    FROM crime_incidents
    GROUP BY suspect_gender, resolution
    ORDER BY incident_count DESC
""")
result.show(truncate=False)

In [0]:
# Saving the table as delta
result.write.mode("overwrite").saveAsTable("crime_incidents_suspect_gender")

In [0]:
# Checking if the table exists
spark.sql("SELECT * FROM crime_incidents_suspect_gender LIMIT 5").show()

## My comments

The **Crime Incidents** dataset from Kaggle proved to be an unusually messy dataset, containing over **200 duplicate rows**, multiple date formats, biologically impossible age values, negative financial figures, and highly inconsistent categorical columns. The scale of data quality issues in a dataset of only **5,250** rows was surprising, suggesting flawed data collection processes — possibly incorrect input forms or ingestion errors. For production use, the remaining columns such as weapon_used and severity would require further standardization before drawing meaningful conclusions. The high null rate in some columns could only be addressed at the data collection stage.

Regarding the SQL analysis (based on closed cases with confirmed arrests only): between **2018** and **2024**, female suspects accounted for 67 incidents vs 60 for male, with average property loss of $27974.71 and $24916.26 respectively. However, 60 records with unknown gender limit the reliability of gender-based conclusions. Given more time, it would be valuable to explore the geographical distribution of crimes using the latitude and longitude data, for instance visualized in Tableau or Power BI.